In [ ]:
import cv2
import io
import numpy as np
import imageio.v2 as imageio
import geopandas as gpd
from PIL import Image, ImageSequence
import matplotlib.pyplot as plt
from rasterio.transform import from_bounds
from shapely.geometry import mapping
import rasterio.features # Required for calculate_district_metrics
import pandas as pd # Import pandas for DataFrame creation
import matplotlib.font_manager as fm
import os
from matplotlib.patches import Patch
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from shapely.geometry import Point
from rasterio.transform import from_bounds
from rasterio.features import geometry_mask
font_paths = fm.findSystemFonts(fontpaths=None, fontext='ttf')
thai_font_path = None
for font_path in font_paths:
    if 'Sarabun' in font_path or 'Garuda' in font_path or 'Laksaman' in font_path: # Look for common Thai fonts
        thai_font_path = font_path
        break

if thai_font_path:
    fm.fontManager.addfont(thai_font_path)
    plt.rcParams['font.family'] = fm.FontProperties(fname=thai_font_path).get_name()
    plt.rcParams['axes.unicode_minus'] = False # Fix minus sign if using unicode font
    print(thai_font_path)
# gif_path = '/content/drive/MyDrive/radar/20260525_040000.webp'
gif_path = r'C:\Users\BMA_01\Documents\ขอข้อมูล\2026-05-01-main-captioner\data\radar_caption\central\history\2026\20260527_020002.webp'
radar_x = 699558.0797  # พิกัด UTM X ของจุดกึ่งกลาง (ใส่ค่าของคุณ)
radar_y = 1530232.3207 # พิกัด UTM Y ของจุดกึ่งกลาง (ใส่ค่าของคุณ)
pixel_resolution = 300     # 1 พิกเซล = กี่เมตร (ตรวจสอบค่านี้อีกครั้ง)
shapefile_path = './../mapdata/Export_Output.shp' # ชื่อไฟล์ Shapefile ของคุณ
threshold = 10
district_names = [
    "ดุสิต",
    "หนองจอก",
    "พระนคร",
    "บางรัก",
    "บางเขน",
    "บางกะปิ",
    "ปทุมวัน",
    "ป้อมปราบศัตรูพ่าย",
    "พระโขนง",
    "มีนบุรี",
    "ลาดกระบัง",
    "ยานนาวา",
    "สัมพันธวงศ์",
    "พญาไท",
    "ธนบุรี",
    "บางกอกใหญ่",
    "ห้วยขวาง",
    "คลองสาน",
    "ตลิ่งชัน",
    "บางกอกน้อย",
    "บางขุนเทียน",
    "ภาษีเจริญ",
    "หนองแขม",
    "ราษฎร์บูรณะ",
    "บางพลัด",
    "ดินแดง",
    "บึงกุ่ม",
    "สาทร",
    "บางซื่อ",
    "จตุจักร",
    "บางคอแหลม",
    "ประเวศ",
    "คลองเตย",
    "สวนหลวง",
    "จอมทอง",
    "ดอนเมือง",
    "ราชเทวี",
    "ลาดพร้าว",
    "วัฒนา",
    "บางแค",
    "หลักสี่",
    "สายไหม",
    "คันนายาว",
    "สะพานสูง",
    "วังทองหลาง",
    "คลองสามวา",
    "บางนา",
    "ทวีวัฒนา",
    "ทุ่งครุ",
    "บางบอน"
]

In [ ]:

target_colors = np.array([
#     [252, 252, 255],
#     [252, 219, 255],
#     [252, 202, 255],
#     [252, 139, 255],
#     [252,   0, 255],
#     [195,   0,  85],
#     [216,   0,  71],
#     [224,   0,  85],
#     [238,   0,   0],
#     [252,  75,   0],
#     [222, 152,   0],
#     [230, 164,   0],
#     [252, 214,   0],
#     [216, 216,   0],
#     [234, 218,   0],
#     [238, 252,   0],
    [  0, 243,   0],
    [  0, 236,   0],
    [  0, 214,  82],
    [  0, 200,   0],
    [  0, 197,   0],
    [  0, 191,   0],
    [  0, 176,   0],
    [  0, 168,   0],
    [  0,   0, 255]
])

values = np.array([
#     66.5, 64.0, 61.5, 
#     59.0, 56.5,
#     54.0, 51.5, 49.0, 46.5,
#     44.0, 41.5, 39.0,
#     36.5, 34.0, 31.5, 29.0,
    26.5, 24.0, 21.5,
    19.0, 16.5, 14.0,
    11.5,
    10.0, 9.5
])

# =========================
# RGB -> HSV
# =========================

target_colors_hsv = cv2.cvtColor(
    target_colors.reshape(-1, 1, 3).astype(np.uint8),
    cv2.COLOR_RGB2HSV
).reshape(-1, 3)

# =========================
# LOAD DATA
# =========================

gif = Image.open(gif_path)

gdf = gpd.read_file(shapefile_path)

frames = [
    frame.copy()
    for frame in ImageSequence.Iterator(gif)
]

# =========================
# PROCESS
# =========================

all_results = []

for frame_idx, frame in enumerate(frames):

    print(f"\nProcessing Frame {frame_idx+1}")

    frame_rgb = frame.convert("RGB")

    img = np.array(frame_rgb)

    height, width = img.shape[:2]

    # =====================
    # IMAGE GEOREFERENCE
    # =====================

    half_width_meter = (
        width * pixel_resolution
    ) / 2

    half_height_meter = (
        height * pixel_resolution
    ) / 2

    xmin = radar_x - half_width_meter
    xmax = radar_x + half_width_meter

    ymin = radar_y - half_height_meter
    ymax = radar_y + half_height_meter

    transform = from_bounds(
        xmin,
        ymin,
        xmax,
        ymax,
        width,
        height
    )

    # =====================
    # RGB -> HSV IMAGE
    # =====================

    hsv_img = cv2.cvtColor(
        img,
        cv2.COLOR_RGB2HSV
    )

    # =====================
    # FIND NEAREST RADAR COLOR
    # =====================

    flat_hsv = hsv_img.reshape(-1, 3)

    # distance matrix
    dist = np.linalg.norm(
        flat_hsv[:, None] -
        target_colors_hsv[None, :],
        axis=2
    )

    nearest_idx = np.argmin(
    dist,
    axis=1
    )

    min_dist = np.min(
        dist,
        axis=1
    )

    COLOR_THRESHOLD = 15

    rain_values = np.where(
        min_dist < COLOR_THRESHOLD,
        values[nearest_idx],
        0
    )
    rain_map = rain_values.reshape(
        height,
        width
    )

    # =====================
    # DISTRICT ANALYSIS
    # =====================

    frame_results = []

    for idx, row in gdf.iterrows():

        district_name = row["DISTRICT_T"]

        geom = [row.geometry]

        district_mask = geometry_mask(
            geom,
            transform=transform,
            invert=True,
            out_shape=(height, width)
        )

        district_rain = rain_map[
            district_mask
        ]

        if len(district_rain) == 0:
            continue

        # remove background
        district_rain = district_rain[
            district_rain > 10
        ]

        if len(district_rain) == 0:

            mean_rain = 0

        else:

            mean_rain = np.mean(
                district_rain
            )

        # severity class

        severity = 0

        if mean_rain >= 45:
            severity = 3

        elif mean_rain >= 30:
            severity = 2

        elif mean_rain >= 10:
            severity = 1

        frame_results.append({
            "district": district_name,
            "mean_rain": round(mean_rain, 2),
            "severity": severity
        })

        all_results.append({
            "frame": frame_idx,
            "district": district_name,
            "mean_rain": round(mean_rain, 2),
            "severity": severity
        })

    # =====================
    # PLOT
    # =====================

    fig, ax = plt.subplots(
        figsize=(12, 12)
    )

    ax.imshow(
        frame_rgb,
        extent=[
            xmin,
            xmax,
            ymin,
            ymax
        ],
        origin="upper"
    )

    # base boundary
    gdf.plot(
        ax=ax,
        facecolor="none",
        edgecolor="white",
        linewidth=0.5
    )

    # =====================
    # HIGHLIGHT DISTRICT
    # =====================

    for result in frame_results:

        district_name = result["district"]

        severity = result["severity"]

        mean_rain = result["mean_rain"]

        if severity == 0:
            continue

        district_geom = gdf[
            gdf["DISTRICT_T"] == district_name
        ]

        # color
        if severity == 1:
            color = "green"

        elif severity == 2:
            color = "orange"

        else:
            color = "red"

        district_geom.plot(
            ax=ax,
            facecolor=color,
            edgecolor="black",
            alpha=0.4
        )

        centroid = district_geom.geometry.centroid.iloc[0]

        ax.text(
            centroid.x,
            centroid.y,
            f"{district_name}\n{mean_rain:.1f}",
            fontsize=7,
            ha="center",
            color="white"
        )

    ax.set_title(
        f"Radar Rainfall Frame {frame_idx+1}"
    )

    plt.show()


In [ ]:
all_results

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split
)

from torchvision import transforms, models
SEED = 43

# random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)
# =========================
# CONFIG
# =========================

CSV_PATH = "train_data.csv"

IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 100
LR = 1e-4

NUM_DISTRICTS = 50
NUM_CLASSES = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
np.random.seed(43)
torch
# =========================
# DATASET
# =========================

class RainDataset(Dataset):

    def __init__(self, csv_path, transform=None):

        self.df = pd.read_csv(csv_path)

        self.transform = transform

        # column 1 = frame image
        self.image_paths = self.df.iloc[:, 1]

        # remove first 2 columns
        self.labels_df = self.df.iloc[:, 2:]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        image_path = self.image_paths.iloc[idx]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            self.labels_df.iloc[idx].values.astype(np.int64)
        )

        return image, labels

# =========================
# TRANSFORM
# =========================

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

# =========================
# FULL DATASET
# =========================

full_dataset = RainDataset(
    CSV_PATH,
    transform=transform
)

# =========================
# SPLIT TRAIN / VALID
# =========================

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

# =========================
# DATALOADER
# =========================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
#     num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
#     num_workers=2
)

# =========================
# MODEL
# =========================

class RainResNet50(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = models.resnet50(
            weights="IMAGENET1K_V1"
        )
        for idx, param in enumerate(self.backbone.parameters()):
            if idx <  60:
                param.requires_grad = False
            # print(idx)
        in_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Linear(
            in_features,
            NUM_DISTRICTS * NUM_CLASSES
        )

    def forward(self, x):

        x = self.backbone(x)

        x = x.view(
            -1,
            NUM_DISTRICTS,
            NUM_CLASSES
        )

        return x
class RainVGG16(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = models.vgg16(
            weights="IMAGENET1K_V1"
        )

        in_features = self.backbone.classifier[6].in_features

        self.backbone.classifier[6] = nn.Linear(
            in_features,
            NUM_DISTRICTS * NUM_CLASSES
        )

    def forward(self, x):

        x = self.backbone(x)

        x = x.view(
            -1,
            NUM_DISTRICTS,
            NUM_CLASSES
        )

        return x
class RainConvNext(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = models.convnext_tiny(
            weights="IMAGENET1K_V1"
        )

        in_features = self.backbone.classifier[2].in_features

        self.backbone.classifier[2] = nn.Linear(
            in_features,
            NUM_DISTRICTS * NUM_CLASSES
        )

    def forward(self, x):

        x = self.backbone(x)

        x = x.view(
            -1,
            NUM_DISTRICTS,
            NUM_CLASSES
        )

        return x
class RainMobileNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = models.mobilenet_v3_large(
            weights="IMAGENET1K_V1"
        )

        in_features = self.backbone.classifier[3].in_features
        for param in self.backbone.parameters():
            param.requires_grad = False
        # replace classifier
        self.backbone.classifier[3] = nn.Sequential(

            nn.Linear(in_features, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(
                512,
                NUM_DISTRICTS * NUM_CLASSES
            )
        )

    def forward(self, x):

        x = self.backbone(x)

        x = x.view(
            -1,
            NUM_DISTRICTS,
            NUM_CLASSES
        )

        return x
class RainEfficientNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = models.efficientnet_b7(
            weights="IMAGENET1K_V1"
        )

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier[1] = nn.Linear(
            in_features,
            NUM_DISTRICTS * NUM_CLASSES
        )

    def forward(self, x):

        x = self.backbone(x)

        x = x.view(
            -1,
            NUM_DISTRICTS,
            NUM_CLASSES
        )

        return x
rain_model = RainResNet50
model = rain_model().to(DEVICE)

# =========================
# LOSS / OPTIMIZER
# =========================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=LR
)

best_val_loss = 999999

# =========================
# TRAIN
# =========================

for epoch in range(EPOCHS):

    # =====================
    # TRAIN
    # =====================

    model.train()

    train_loss = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = 0

        for i in range(NUM_DISTRICTS):

            loss += criterion(
                outputs[:, i, :],
                labels[:, i]
            )

        loss = loss / NUM_DISTRICTS

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # =====================
    # VALIDATION
    # =====================

    model.eval()

    val_loss = 0

    total_correct = 0
    total_count = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            loss = 0

            for i in range(NUM_DISTRICTS):

                loss += criterion(
                    outputs[:, i, :],
                    labels[:, i]
                )

            loss = loss / NUM_DISTRICTS

            val_loss += loss.item()

            # =====================
            # ACCURACY
            # =====================

            preds = outputs.argmax(dim=-1)

            correct = (
                preds == labels
            ).sum().item()

            total_correct += correct
            total_count += labels.numel()

    val_loss /= len(val_loader)

    val_acc = total_correct / total_count

    # =====================
    # SAVE BEST MODEL
    # =====================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "nbest_rain_model.pth"
        )

        print("Best model saved")

    # =====================
    # PRINT
    # =====================

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Val Accuracy: {val_acc:.4f}"
    )

    print("-" * 50)

In [ ]:
# =========================
# RAIN LEVEL NAMES
# =========================

rain_level_names = [
    "ไม่มีฝน",
    "ฝนเล็กน้อย",
    "ฝนปานกลาง",
    "ฝนหนัก"
]

# =========================
# DISTRICT NAMES
# =========================

district_names = list(
    full_dataset.labels_df.columns
)

# =========================
# TEST VALID DATASET
# =========================

model.eval()

# ANSI COLOR
GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"

print("=" * 100)
print("VALIDATION SAMPLE PREDICTION")
print("=" * 100)
all_preds = []
all_labels = []
with torch.no_grad():

    for sample_idx in range(len(val_dataset)):
        
        # =====================
        # GET SAMPLE
        # =====================

        image, labels = val_dataset[sample_idx]

        image_tensor = image.unsqueeze(0).to(DEVICE)

        labels = labels.numpy()

        # =====================
        # SHOW IMAGE
        # =====================

        image_np = image.permute(1, 2, 0).numpy()
        print("=" * 100)
        plt.figure(figsize=(4, 4))

        plt.imshow(image_np)

        plt.axis("off")

        plt.title(f"Validation Sample {sample_idx}")

        plt.show()

        # =====================
        # PREDICT
        # =====================

        outputs = model(image_tensor)

        preds = outputs.argmax(dim=-1)

        preds = preds[0].cpu().numpy()
        
        all_preds.extend(
            preds.flatten().tolist()
        )

        all_labels.extend(
            labels.flatten().tolist()
        )
        # =====================
        # IMAGE PATH
        # =====================

        real_idx = val_dataset.indices[sample_idx]

        image_path = full_dataset.image_paths.iloc[
            real_idx
        ]

        print("\n")
        
        print(f"{YELLOW}IMAGE: {image_path}{RESET}")
#         print("=" * 100)

        # =====================
        # PRINT COMPARE
        # =====================

        for i in range(NUM_DISTRICTS):

            actual_class = int(labels[i])

            pred_class = int(preds[i])

            # print only rain district
            if actual_class > 0 or pred_class > 0:

                if actual_class == pred_class:

                    color = GREEN
                    status = "CORRECT"

                else:

                    color = RED
                    status = "WRONG"

                print(
                    color +
                    f"{district_names[i]:20}"
                    f"| actual: {rain_level_names[actual_class]:12}"
                    f"| pred: {rain_level_names[pred_class]:12}"
                    f"| {status}"
                    + RESET
                )
print("\n")
print("=" * 100)
print("METRICS")
print("=" * 100)

precision = precision_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

print(
    f"Recall    : {recall:.4f} "
    f"(ความสามารถในการตรวจจับเหตุการณ์ฝนจริง "
    f"ยิ่งสูงยิ่งตรวจพบฝนได้ครบถ้วน)"
)
print(
    f"Precision : {precision:.4f} "
    f"(เมื่อทำนายว่าฝนตก มีความถูกต้องกี่ %)"
)
print(f"F1 Score  : {f1:.4f}")
print("\n")
print("=" * 100)
print("CLASSIFICATION REPORT")
print("=" * 100)

# print(
#     classification_report(
#         all_labels,
#         all_preds,
#         target_names=rain_level_names,
#         zero_division=0
#     )
# )

In [ ]:
# import requests
# from PIL import Image
# from io import BytesIO
# import matplotlib.pyplot as plt

# # Replace with your image URL

# try:

#     img = Image.open('id01.jpg')

#     plt.figure(figsize=(16, 16))
#     plt.imshow(img)
#     plt.title('Image from URL')
#     plt.axis('off')
#     plt.show()
# except requests.exceptions.RequestException as e:
#     print(f"Error fetching image from URL: {e}")
# except Exception as e:
#     print(f"Error processing image: {e}")